# TTA vs RAG on Adversarial Temporal Tasks

This notebook runs the **adversarial temporal benchmark** (reversion, interval, causal, multi-reversion, interval-midpoint, multi-entity join) and compares:
- **A** (plain RAG: retrieve top-k, pick latest)
- **D** (full TTA: validity + decay + tiebreak)
- **D_revised** (TTA validity-only + confidence tiebreak, no decay)

Goal: show that TTA handles temporal reversion and interval constraints better than naive RAG.

## 1. Setup

Clone the repo (or upload the project) and install dependencies.

In [ ]:
# Clone repo (public). If you have the repo locally, upload it and skip this.
!git clone https://github.com/toxzak-svg/time.git /content/time 2>/dev/null || true
%cd /content/time
print("✓ Working in repo root")

In [ ]:
# No extra deps needed for adversarial benchmark (no sentence-transformers)
print("✓ Ready")

## 2. Generate Adversarial Data

Creates facts and questions for: **Reversion**, **Interval**, **CausalReasoning**, **MultiReversion**, **IntervalMidpoint**, **MultiEntityJoin**.

In [ ]:
!python scripts/generate_adversarial_temporal.py --out-dir benchmarks \
  --reversion 15 --interval 10 --causal 10 \
  --multi-reversion 5 --interval-midpoint 5 --multi-entity 5 --seed 42

## 3. Run Adversarial Benchmark (A vs D vs D_revised)

In [ ]:
!python scripts/run_adversarial_benchmark.py \
  --facts benchmarks/adversarial_temporal_facts.jsonl \
  --questions benchmarks/adversarial_temporal_questions.jsonl \
  --systems A,D,D_revised \
  --out results/adversarial_colab_results.csv

## 4. Results Table and Conclusion

In [ ]:
import pandas as pd
from pathlib import Path

path = Path("results/adversarial_colab_results.csv")
if path.exists():
    df = pd.read_csv(path)
    display(df)
    print("\n--- Summary ---")
    for _, row in df.iterrows():
        sys_name = {"A": "RAG (plain)", "D": "TTA (full)", "D_revised": "TTA (validity-only)"}.get(row["system"], row["system"])
        print(f"{sys_name}: OverallAccuracy = {row.get('OverallAccuracy', 'N/A')}")
else:
    print("Run the benchmark cell first.")

In [ ]:
# Per-task comparison (TTA vs RAG)
if path.exists():
    a = df[df["system"] == "A"].iloc[0]
    d = df[df["system"] == "D"].iloc[0]
    dr = df[df["system"] == "D_revised"].iloc[0]
    cols = [c for c in df.columns if c.endswith("Accuracy") and c != "OverallAccuracy"]
    print("Task family accuracy: RAG (A) vs TTA (D) vs TTA revised (D_revised)\n")
    for c in sorted(cols):
        print(f"  {c}: A={a.get(c, 0):.2f}  D={d.get(c, 0):.2f}  D_revised={dr.get(c, 0):.2f}")
    print("\nConclusion: TTA (D or D_revised) should outperform RAG (A) on Reversion and Interval tasks.")